# PromptSec-FM — XLM-R multitâche v0.2

Workflow scientifique isolé. Il ne modifie ni v0.1, ni la taxonomie gelée, ni les 6 000 records SILVER. L’entraînement complet est désactivé par défaut.

## 1. Paramètres sûrs

Les sorties v0.2 sont distinctes. Le français OOD ne sert jamais à la sélection.

In [ ]:
REPOSITORY_URL = "https://github.com/LIGHTER91/PromptSec-FM.git"
REPOSITORY_REF = "main"
REPO_DIR = "/content/PromptSec-FM"
DRIVE_ROOT = "/content/drive/MyDrive/PromptSec-FM"
SMOKE_CHECKPOINT_ROOT = "/content/promptsec_smoke/checkpoints"
SMOKE_REPORT_ROOT = "/content/promptsec_smoke/reports"
DATASET_ARCHIVE = f"{DRIVE_ROOT}/data/policybench-codex-v0.1-colab.zip"
DATASET_DIR = "/content/promptsec_data/policybench-codex-v0.1"
V0_1_REPORT_ROOT = f"{DRIVE_ROOT}/reports/xlmr-base-multitask-v0.1"
START_V0_2_TRAINING = False
RUN_SMOKE_TEST_FIRST = True
RESET_SMOKE_TEST = True
RESUME_SMOKE_TEST = False
RESUME = True
RUN_BALANCED_EXPERIMENT = True
RUN_RELATIONAL_EXPERIMENT = True
RUN_FOCAL_EXPERIMENT = False
BILINGUAL_FINAL_SILVER_MODEL = False
PRUNE_CHECKPOINTS = False

## 2. Montage de Drive

Drive contient l’archive vérifiée et les artefacts.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Clone complet

Le dépôt complet est cloné; aucun sous-ensemble manuel.

In [ ]:
import subprocess
from pathlib import Path

if not Path(REPO_DIR).exists():
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_REF, REPOSITORY_URL, REPO_DIR],
        check=True,
    )
commit = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_DIR,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print("Commit utilisé:", commit)

## 4. Installation

Installation éditable avec le groupe `training`.

In [ ]:
import importlib
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".[training]"],
    cwd=REPO_DIR,
    check=True,
)
expected_source = (Path(REPO_DIR) / "src").resolve()
expected_source_text = str(expected_source)
if expected_source_text not in sys.path:
    sys.path.insert(0, expected_source_text)
importlib.invalidate_caches()

training_module = importlib.import_module("promptsec.training.dataset")
training_path = Path(training_module.__file__).resolve()
if expected_source not in training_path.parents:
    raise RuntimeError(f"Import PromptSec inattendu: {training_path}")
load_training_dataset = training_module.load_training_dataset
print("Import PromptSec vérifié depuis:", training_path)

## 5. Vérification de l’archive

Le SHA-256 et le manifeste Colab sont vérifiés.

In [ ]:
import hashlib
import zipfile

archive = Path(DATASET_ARCHIVE)
if not archive.is_file():
    raise FileNotFoundError(archive)
print("SHA-256 archive:", hashlib.sha256(archive.read_bytes()).hexdigest())
extract_root = Path(DATASET_DIR).parent
if not Path(DATASET_DIR).exists():
    extract_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as stream:
        stream.extractall(extract_root)

## 6. Validation des 6 000 records

Le chargeur bloque tout écart SILVER, split ou checksum.

In [ ]:
bundle = load_training_dataset(DATASET_DIR)
print(bundle.integrity_report)
assert bundle.integrity_report["records"] == 6000
assert bundle.integrity_report["automatic_gold_records"] == 0

## 7. Audit v0.1

Les poids v0.1 ne sont pas nécessaires à la comparaison.

In [ ]:
from promptsec.training.diagnostics_v02 import v01_diagnostic_audit

v01_audit = v01_diagnostic_audit(V0_1_REPORT_ROOT)
print(v01_audit["status"], v01_audit.get("available_files", []))

## 8. Audit contrefactuel v0.2

Seules les paires complètes du train officiel sont éligibles.

In [ ]:
from promptsec.training.pairing import audit_counterfactual_groups

pair_audit = audit_counterfactual_groups(bundle.records_by_split)
print(pair_audit["splits"].get("train"))
print(pair_audit["splits"].get("validation"))

## 9. Distributions

La catégorie sert au sampling et au diagnostic, jamais à l’entrée modèle.

In [ ]:
from promptsec.training.diagnostics_v02 import (
    category_distribution,
    class_and_multilabel_audit,
)

print(category_distribution(bundle.records_by_split["train"]))
training_audit = class_and_multilabel_audit(bundle.records_by_split["train"], bundle.mappings)
print(training_audit["heads"])

## 10. Commandes communes

Toutes les commandes appellent le CLI testé.

In [ ]:
CONFIG = "configs/xlmr_multitask_colab_v0.2.yaml"


def run_command(command):
    print("Commande:", command)
    process = subprocess.Popen(
        command,
        cwd=REPO_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        shell=False,
    )
    output_tail = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        output_tail.append(line.rstrip())
        output_tail = output_tail[-80:]
    return_code = process.wait()
    if return_code:
        tail = "\n".join(output_tail)
        raise RuntimeError(
            f"Commande échouée avec le code {return_code}.\nDernières lignes du processus:\n{tail}"
        )


def experiment_command(name, *, resume=True, smoke=False):
    checkpoint_root = SMOKE_CHECKPOINT_ROOT if smoke else f"{DRIVE_ROOT}/checkpoints"
    report_root = SMOKE_REPORT_ROOT if smoke else f"{DRIVE_ROOT}/reports"
    command = [
        sys.executable,
        "scripts/train_xlmr_multitask.py",
        "--config",
        CONFIG,
        "--dataset",
        DATASET_DIR,
        "--output",
        f"{checkpoint_root}/xlmr-base-multitask-v0.2-{name}",
        "--reports",
        f"{report_root}/xlmr-base-multitask-v0.2-{name}",
        "--experiment",
        name,
        "--v0-1-report-root",
        V0_1_REPORT_ROOT,
        "--resume" if resume else "--no-resume",
    ]
    if PRUNE_CHECKPOINTS:
        command.append("--prune-checkpoints")
    return command

## 11. Smoke propre Experiment B

Par défaut il démarre propre et sans reprise.

In [ ]:
if RUN_SMOKE_TEST_FIRST:
    smoke = experiment_command("relational", resume=RESUME_SMOKE_TEST, smoke=True)
    smoke += [
        "--smoke-test",
        "--max-train-records",
        "32",
        "--max-validation-records",
        "16",
        "--epochs",
        "1",
        "--max-length",
        "128",
    ]
    if RESET_SMOKE_TEST:
        smoke.append("--reset-smoke-test")
    run_command(smoke)

## 12. Garde d’exécution

Le smoke peut tourner; les runs complets exigent une activation explicite.

In [ ]:
print("START_V0_2_TRAINING =", START_V0_2_TRAINING)

## 13. Experiment A — Balanced

Aucune perte relationnelle; calibration multilabel validation-only.

In [ ]:
if START_V0_2_TRAINING and RUN_BALANCED_EXPERIMENT:
    run_command(experiment_command("balanced", resume=RESUME))

## 14. Experiment B — Relational

Paires, JS tête par tête et cohérence structurée du verdict.

In [ ]:
if START_V0_2_TRAINING and RUN_RELATIONAL_EXPERIMENT:
    run_command(experiment_command("relational", resume=RESUME))

## 15. Experiment C — Focal optionnel

L’asymmetric focal BCE est désactivée par défaut.

In [ ]:
if START_V0_2_TRAINING and RUN_FOCAL_EXPERIMENT:
    run_command(experiment_command("focal", resume=RESUME))

## 16. Évaluation sélectionnée

Les tests officiels ne sont évalués qu’après le checkpoint validation.

In [ ]:
print("Les rapports test_metrics.json sont écrits uniquement après la sélection validation.")

## 17. Diagnostics relationnels

Affichage sans payload : contrefactuels, sur-défense, logique et seuils.

In [ ]:
import json


def show_report(root, name):
    path = Path(root) / name
    print(name, json.loads(path.read_text()) if path.is_file() else "indisponible")


selected_reports = f"{DRIVE_ROOT}/reports/xlmr-base-multitask-v0.2-relational"
for name in [
    "counterfactual_results.json",
    "hard_negative_results.json",
    "multilabel_thresholds.json",
    "logical_consistency_results.json",
]:
    show_report(selected_reports, name)

## 18. Diagnostics EN/FR

Le français OOD est consulté seulement après sélection; les splits diffèrent au-delà de la langue.

In [ ]:
show_report(selected_reports, "language_results.json")

## 19. Vérification des exports

Les checksums de `best_model` doivent être valides avant tout pruning.

In [ ]:
from promptsec.training.retention import verify_export

best_export = f"{DRIVE_ROOT}/checkpoints/xlmr-base-multitask-v0.2-relational/best_model"
if Path(best_export).is_dir():
    print(verify_export(best_export))

## 20. Rétention optionnelle

Le défaut reste un dry-run; v0.1 est hors périmètre par construction.

In [ ]:
print("PRUNE_CHECKPOINTS =", PRUNE_CHECKPOINTS)
print("Le CLI écrit checkpoint_pruning_manifest.json après vérification de l’export.")

## 21. Reproduction exacte

Copiez les listes ci-dessous; aucune commande v0.1 n’est relancée.

In [ ]:
print(
    "Smoke:",
    experiment_command("relational", resume=False, smoke=True)
    + ["--smoke-test", "--reset-smoke-test"],
)
print("A:", experiment_command("balanced", resume=True))
print("B:", experiment_command("relational", resume=True))
print("C optionnel:", experiment_command("focal", resume=True))